# dataset_rerank 全流程

> 请先确认：
> - API token/endpoint 已更新为可用值
> - 目标 collection 是否允许覆盖

## 1. 配置数据路径（测试集 mergebert_20，训练库 mergebert_80_cr_train）

In [ ]:
import json
from pathlib import Path

# 固定目录：训练集用于入库，测试集用于推理
TRAIN_JSON_DIR = Path("dataset/after/mergebert_80_cr_train")
TEST_JSON_DIR = Path("dataset/mergebert_20")
WORK_ROOT = Path("dataset_split")
TRAIN_WORK_DIR = WORK_ROOT / "train"
TEST_WORK_DIR = WORK_ROOT / "test"
VECTOR_DIR = WORK_ROOT / "vector"
QWEN_OUT_DIR = WORK_ROOT / "outputs/qwen"

TRAIN_WORK_DIR.mkdir(parents=True, exist_ok=True)
TEST_WORK_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_DIR.mkdir(parents=True, exist_ok=True)
QWEN_OUT_DIR.mkdir(parents=True, exist_ok=True)

if not TRAIN_JSON_DIR.exists():
    raise FileNotFoundError(f"missing train dir: {TRAIN_JSON_DIR}")
if not TEST_JSON_DIR.exists():
    raise FileNotFoundError(f"missing test dir: {TEST_JSON_DIR}")

train_files = sorted(TRAIN_JSON_DIR.glob("*_time.json"))
test_files = sorted(TEST_JSON_DIR.glob("*_time20.json"))

print(f"train source dir: {TRAIN_JSON_DIR} -> {len(train_files)} files")
print(f"test source dir: {TEST_JSON_DIR} -> {len(test_files)} files")
print(f"vector output dir: {VECTOR_DIR}")
print(f"qwen output dir: {QWEN_OUT_DIR}")

if not train_files:
    raise ValueError("no *_time.json files found in mergebert_80_cr_train")
if not test_files:
    raise ValueError("no *_time20.json files found in mergebert_20")

## 2. 训练JSON转vector文件（来自 mergebert_80_cr_train）

In [ ]:
import json
from pathlib import Path

def json_to_vector(input_path, output_path):
    with open(input_path, "r", encoding="utf-8") as file:
        data_list = json.load(file)

    valid_records_count = 0
    with open(output_path, "w", encoding="utf-8") as output_file:
        for data in data_list:
            if not isinstance(data, dict):
                continue
            res_label = data.get("res_label", "")
            if res_label == "null":
                continue
            a_contents = data.get("a_contents", "")
            o_contents = data.get("o_contents", "")
            b_contents = data.get("b_contents", "")
            res_region = data.get("res_region", "")
            global_id = data.get("id", "")

            conflict = (
                "concurrent_change\n" + a_contents +
                "\nbase_content\n" + o_contents +
                "\nincoming_change\n" + b_contents
            )
            processed_conflict = conflict.replace("\n", " _newline_ ").replace("\t", " ")
            processed_res = str(res_region).replace("\n", " _newline_ ").replace("\t", " ")
            output_file.write(processed_conflict + "\t" + processed_res + "\t" + str(global_id) + "\n")
            valid_records_count += 1

    print(f"processed {input_path} -> {output_path} ({valid_records_count} rows)")

# 使用第1步中定义的 TRAIN_JSON_DIR / VECTOR_DIR
json_candidates = sorted(TRAIN_JSON_DIR.glob("*_time.json"))
if not json_candidates:
    print(f"no json files found under {TRAIN_JSON_DIR}")
else:
    for json_path in json_candidates:
        output_path = VECTOR_DIR / f"{json_path.stem}_Conflicts.txt"
        json_to_vector(json_path, output_path)

## 3. 向量入库（按训练文件名建 collection）

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import torch
from pymilvus import MilvusClient, DataType, Function, FunctionType
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT"
TOKEN = "YOUR_MILVUS_TOKEN"

DROP_EXISTING = False
GROUP_ID = 98 
# dataset_rerank group id 99

client = MilvusClient(uri=CLUSTER_ENDPOINT, token=TOKEN)

def safe_collection_name(name):
    return ''.join(ch if ch.isalnum() or ch == '_' else '_' for ch in str(name))

def ensure_collection(collection_name):
    collection_name = safe_collection_name(collection_name)
    if collection_name in client.list_collections():
        if DROP_EXISTING:
            client.drop_collection(collection_name=collection_name)
        else:
            return
    schema = MilvusClient.create_schema(auto_id=True, enable_dynamic_field=True)
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    schema.add_field(field_name="global_id", datatype=DataType.INT32)
    schema.add_field(field_name="chunk_content", datatype=DataType.VARCHAR, max_length=2048, enable_analyzer=True)
    schema.add_field(field_name="chunk_resolution", datatype=DataType.VARCHAR, max_length=1024)
    schema.add_field(field_name="chunk_sparse_embedding", datatype=DataType.SPARSE_FLOAT_VECTOR)
    schema.add_field(field_name="chunk_embedding", datatype=DataType.FLOAT_VECTOR, dim=768)

    bm25_function = Function(
        name="text_bm25_emb",
        input_field_names="chunk_content",
        output_field_names="chunk_sparse_embedding",
        function_type=FunctionType.BM25,
    )
    schema.add_function(bm25_function)

    index_params = MilvusClient.prepare_index_params()
    index_params.add_index(
        field_name="chunk_embedding",
        metric_type="L2",
        index_type="AUTOINDEX",
        index_name="vector_index",
    )
    index_params.add_index(
        field_name="chunk_sparse_embedding",
        index_name="sparse_inverted_index",
        index_type="SPARSE_INVERTED_INDEX",
        metric_type="BM25",
        params={"drop_ratio_build": 0.2},
    )

    client.create_collection(collection_name=collection_name, schema=schema, index_params=index_params)

def load_model(local_model_path="./codeT5/codet5-base"):
    try:
        tokenizer = AutoTokenizer.from_pretrained(local_model_path)
        model = AutoModelForSeq2SeqLM.from_pretrained(local_model_path).to("cuda" if torch.cuda.is_available() else "cpu")
        print("using local CodeT5 model")
    except Exception as e:
        print(f"local model load failed: {e}")
        tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-base")
        model = AutoModelForSeq2SeqLM.from_pretrained("Salesforce/codet5-base").to("cuda" if torch.cuda.is_available() else "cpu")
        tokenizer.save_pretrained(local_model_path)
        model.save_pretrained(local_model_path)
        print("downloaded CodeT5 model")
    return tokenizer, model

tokenizer, model = load_model()

def insert_vector_file(vector_path):
    repo_name = safe_collection_name(vector_path.stem.replace("_Conflicts", ""))
    ensure_collection(repo_name)

    pair = pd.read_csv(vector_path, sep="\t", header=None).drop_duplicates()
    if pair.shape[1] < 3:
        raise ValueError(f"invalid vector file: {vector_path}")
    conflicts = pair[0].fillna("").values
    res_labels = pair[1].fillna("").values
    global_ids = pair[2].fillna(0).astype(int).values

    data_pairs = list(zip(conflicts, res_labels, global_ids))
    for conflict, resolution, g_id in tqdm(data_pairs, desc=f"{repo_name}"):
        inputs = tokenizer(conflict, padding=True, truncation=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            enc = model.encoder(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
            embeddings = enc.last_hidden_state.mean(dim=1).cpu().numpy()
        encode = embeddings[0]

        res_data = resolution if isinstance(resolution, str) and resolution.strip() else "_newline_"
        client.insert(
            collection_name=repo_name,
            data=[
                {
                    "group_id": GROUP_ID,
                    "global_id": int(g_id),
                    "chunk_content": str(conflict).encode("utf-8")[:2048].decode("utf-8", errors="ignore"),
                    "chunk_resolution": str(res_data).encode("utf-8")[:1024].decode("utf-8", errors="ignore"),
                    "chunk_embedding": encode,
                }
            ],
        )
    print(f"inserted {len(data_pairs)} rows into {repo_name}")

vector_dir = Path("dataset_split/vector")
vector_files = sorted(vector_dir.glob("*_Conflicts.txt"))
if not vector_files:
    print("no vector files found under dataset_split/vector")
else:
    for vector_file in vector_files:
        insert_vector_file(vector_file)

## 4. 生成三类输出（分模型调用 API）

In [ ]:
import json
from difflib import SequenceMatcher
from pathlib import Path
from colorama import Fore, Style
from utils import simplePrompt, prompt1
from alibaba_api import qwen
from deepseek_api import deepSeek
from openai_api import gpt_35
from database_api_T5_BM25_L2 import query_similar

CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT"
TOKEN = "YOUR_MILVUS_TOKEN"

def sim(a, b):
    if not a or not b:
        return 0
    return SequenceMatcher(None, a, b).ratio()

def contains_tokens(source, target):
    return any(tok.strip() and tok in target for tok in source.split())

def normalize_refer_conflict(raw_conflict):
    return (
        raw_conflict.replace(" _newline_ ", "\n")
        .replace("concurrent_change", "<<<<<<<")
        .replace("base_content", "|||||||")
        .replace("incoming_change", "=======")
        + "\n>>>>>>>"
    )

def strip_markers(text):
    return (
        text.replace("<<<<<<<", "")
        .replace("|||||||", "")
        .replace("=======", "")
        .replace(">>>>>>>", "")
    )

def score_reference(refer_conflict, a, b, o):
    score = 0
    base_text = strip_markers(refer_conflict)
    score += 0.5 * sim(base_text, a)
    score += 0.5 * sim(base_text, b)
    score -= 0.7 * sim(base_text, o)
    if contains_tokens(a, base_text) and contains_tokens(b, base_text):
        score += 1.0
    elif contains_tokens(a, base_text) or contains_tokens(b, base_text):
        score += 0.3
    score -= abs(len(base_text) - (len(a) + len(b))) * 0.0001
    return score

def pick_best_reference(related_cs, related_rs, a, b, o, current_id, global_ids):
    candidates = []
    for idx, raw_c in enumerate(related_cs):
        if global_ids[idx] == current_id:
            continue
        refer_c = normalize_refer_conflict(raw_c)
        refer_r = related_rs[idx].replace("_newline_", "\n")
        candidates.append((refer_c, refer_r))
    if not candidates:
        return "", ""
    ranked = sorted(
        candidates,
        key=lambda pair: score_reference(pair[0], a, b, o),
        reverse=True,
    )
    return ranked[0]

def map_test_to_train_collection(input_path):
    # mergebert_20 下测试文件：xxx_time20.json
    # mergebert_80_cr_train 下训练 collection：xxx_time
    stem = Path(input_path).stem
    if stem.endswith("_time20"):
        mapped = stem[:-2]  # 去掉末尾 20 -> *_time
    else:
        mapped = stem
    # 处理命名差异：spring_time20 -> spring-boot_time
    if mapped == "spring_time":
        mapped = "spring-boot_time"
    return safe_collection_name(mapped)

def load_existing_results(output_path):
    path = Path(output_path)
    if not path.exists():
        return None
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

def merge_existing_results(input_data, existing_data, id_key="id"):
    if not existing_data:
        return input_data
    existing_by_id = {item.get(id_key): item for item in existing_data if isinstance(item, dict)}
    merged = []
    for item in input_data:
        if not isinstance(item, dict):
            merged.append(item)
            continue
        existing = existing_by_id.get(item.get(id_key))
        if existing:
            merged.append({**item, **existing})
        else:
            merged.append(item)
    return merged

def find_resume_index(data, required_fields):
    for i, item in enumerate(data):
        if not isinstance(item, dict):
            return i
        missing = any(field not in item or not item[field] for field in required_fields)
        if missing:
            return i
    return len(data)

def request_model(input_path, output_path, model_func, num_candidates=3, top_k_similar=5, sample_limit=None, start_index=0, resume=True):
    required_fields = ["line_no_T5", "line_withRefer_T5", "line_withrerankT5"]
    json_data = []
    try:
        with open(input_path, "r", encoding="utf-8") as input_file:
            json_data = json.load(input_file)

        if not isinstance(json_data, list):
            raise ValueError(f"input json must be list: {input_path}")

        existing = load_existing_results(output_path) if resume else None
        json_data = merge_existing_results(json_data, existing)

        collection_name = map_test_to_train_collection(input_path)
        print(f"use collection: {collection_name} for test file: {Path(input_path).name}")

        processed = 0
        if resume and start_index == 0:
            start_index = find_resume_index(json_data, required_fields)
            if start_index >= len(json_data):
                print("all items already processed, nothing to do")
                return

        for i, item in enumerate(json_data):
            if i < start_index:
                continue
            if sample_limit is not None and processed >= sample_limit:
                break
            if not isinstance(item, dict):
                continue

            current_id = item.get("id", "")
            res_region = item.get("res_region", "")

            conflict = (
                "concurrent_change\n" + item.get("a_contents", "") +
                "\nbase_content\n" + item.get("o_contents", "") +
                "\nincoming_change\n" + item.get("b_contents", "")
            )
            diff_conflict = conflict.replace("concurrent_change", "<<<<<<<").replace("base_content", "|||||||").replace("incoming_change", "=======") + "\n>>>>>>>"
            query = conflict.replace("\n", " _newline_ ").replace("\t", " ")

            global_ids, related_cs, related_rs, _ = query_similar(CLUSTER_ENDPOINT, TOKEN, collection_name, query, top_k_similar)

            if not item.get("line_no_T5"):
                item["line_no_T5"] = model_func(question=simplePrompt(diff_conflict), number=num_candidates, truth=res_region)

            if not item.get("line_withRefer_T5"):
                refer_c1, refer_r1 = "", ""
                for j in range(len(global_ids)):
                    if global_ids[j] != current_id:
                        refer_c1 = normalize_refer_conflict(related_cs[j])
                        refer_r1 = related_rs[j].replace("_newline_", "\n")
                        break
                item["line_withRefer_T5"] = model_func(question=prompt1(refer_c1, refer_r1, diff_conflict), number=num_candidates, truth=res_region)

            if not item.get("line_withrerankT5"):
                best_c, best_r = pick_best_reference(related_cs, related_rs, item.get("a_contents", ""), item.get("b_contents", ""), item.get("o_contents", ""), current_id, global_ids)
                item["line_withrerankT5"] = model_func(question=prompt1(best_c, best_r, diff_conflict), number=num_candidates, truth=res_region)

            processed += 1
            if processed % 10 == 0:
                with open(output_path, "w", encoding="utf-8") as out:
                    json.dump(json_data, out, indent=2, ensure_ascii=False)
                print(f"{Fore.GREEN}auto-save {processed} items{Style.RESET_ALL}")

    except Exception as e:
        print(f"{Fore.RED}Error: {e}{Style.RESET_ALL}")
    finally:
        with open(output_path, "w", encoding="utf-8") as out:
            json.dump(json_data, out, indent=2, ensure_ascii=False)
        print(f"{Fore.GREEN}saved to {output_path}{Style.RESET_ALL}")

In [ ]:
import importlib
import database_api_T5_BM25_L2 as retrieval_api

retrieval_api = importlib.reload(retrieval_api)
query_similar = retrieval_api.query_similar
print("reloaded retrieval module with cached Milvus client and CodeT5 model")

### 4.1 Qwen（批量跑 mergebert_20 各测试文件）

In [ ]:
# 批量跑 mergebert_20 全量测试集，自动复用已有输出并从断点继续
test_inputs = sorted(TEST_JSON_DIR.glob("*_time20.json"))
if not test_inputs:
    print(f"no test files found in {TEST_JSON_DIR}")
else:
    for test_path in test_inputs:
        out_path = QWEN_OUT_DIR / f"{test_path.stem}_qwen.json"
        print(f"\n=== qwen run: {test_path.name} -> {out_path.name} ===")
        request_model(
            input_path=str(test_path),
            output_path=str(out_path),
            model_func=qwen,
            num_candidates=5,
            top_k_similar=5,
            sample_limit=None,
            start_index=0,
            resume=True,
        )

In [ ]:
request_model(
    input_path="dataset_split/test/mergebert_20.json",
    output_path="dataset_split/outputs/qwen/mergebert_20_qwen_buchong.json",
    model_func=qwen,
    num_candidates=5,
    top_k_similar=5,
    sample_limit=11111111,
    start_index=0,
)

### 4.2 DeepSeek

In [ ]:
request_model(
    input_path="dataset_split/test/mergebert_20.json",
    output_path="dataset_split/outputs/deepseek/mergebert_20_deepseek.json",
    model_func=deepSeek,
    num_candidates=5,
    top_k_similar=5,
    sample_limit=500,
    start_index=0,
)

### 4.3 GPT

In [ ]:
request_model(
    input_path="dataset_rerank/test/merged_bottom20.json",
    output_path="dataset_rerank/outputs/gpt/merged_bottom20_gpt.json",
    model_func=gpt_35,
    num_candidates=3,
    top_k_similar=5,
    sample_limit=500,
    start_index=0,
)

## 5. 生成后的二次重排序（单独执行）

In [ ]:
import json
from difflib import SequenceMatcher
from pathlib import Path

def sim(a, b):
    if not a or not b:
        return 0
    return SequenceMatcher(None, a, b).ratio()

def contains_tokens(source, target):
    return any(tok.strip() and tok in target for tok in str(source).split())

def score_candidate(candidate, a, b, o):
    candidate_str = str(candidate)
    a = str(a or "")
    b = str(b or "")
    o = str(o or "")

    score = 0.0
    score += 0.5 * sim(candidate_str, a)
    score += 0.5 * sim(candidate_str, b)
    score -= 0.7 * sim(candidate_str, o)
    if contains_tokens(a, candidate_str) and contains_tokens(b, candidate_str):
        score += 2
    elif contains_tokens(a, candidate_str) or contains_tokens(b, candidate_str):
        score += 0.5
    score -= abs(len(a) - len(candidate_str)) * 0.0005
    return score

def rerank_json_file_inplace(input_file, fields):
    try:
        with open(input_file, "r", encoding="utf-8") as f:
            dataset = json.load(f)
    except Exception as e:
        print(f"[skip] failed to load: {input_file} ({e})")
        return False

    if not isinstance(dataset, list):
        print(f"[skip] not a list dataset: {input_file}")
        return False

    changed = False
    for item in dataset:
        if not isinstance(item, dict):
            continue
        a = item.get("a_contents", "")
        b = item.get("b_contents", "")
        o = item.get("o_contents", "")
        for field in fields:
            candidates = item.get(field)
            if not isinstance(candidates, list) or len(candidates) <= 1:
                continue
            item[field] = sorted(
                candidates,
                key=lambda c: score_candidate(c, a, b, o),
                reverse=True,
            )
            changed = True

    if not changed:
        return True

    with open(input_file, "w", encoding="utf-8") as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)

    return True

output_root = Path("produce/mergebert_20/rerank")
fields = [
    "line_no",
    "line_withRefer",
    "line_withRefer1",
    "line_withRefer2",
    "line_withRefer3",
    "line_withRefer4",
    "line_withRefer5",
]

json_files = sorted("dataset_split/outputs/qwen")
processed = 0
skipped = 0

for i, input_path in enumerate(json_files, start=1):
    ok = rerank_json_file_inplace(input_path, fields)
    if ok:
        processed += 1
    else:
        skipped += 1
    if i % 50 == 0 or i == len(json_files):
        print(f"progress {i}/{len(json_files)} (processed={processed}, skipped={skipped})")

print(f"done. processed={processed}, skipped={skipped}, root={output_root}")

## 6. 精准匹配评估与summary.md

In [ ]:
import json
from pathlib import Path

def exact_match_hit(candidates, truth):
    if truth is None:
        return False
    truth_str = str(truth).strip()
    if not truth_str:
        return False
    if isinstance(candidates, list):
        return any(str(c).strip() == truth_str for c in candidates)
    return str(candidates).strip() == truth_str

def extract_records(data):
    if isinstance(data, list):
        return [item for item in data if isinstance(item, dict)]
    if isinstance(data, dict):
        for key in ("data", "items", "records", "rows"):
            value = data.get(key)
            if isinstance(value, list):
                return [item for item in value if isinstance(item, dict)]
    return []

def evaluate_file(records, fields):
    results = {field: {"hit": 0, "total": 0} for field in fields}
    fields_present = set()
    for item in records:
        truth = item.get("res_region", "")
        for field in fields:
            if field not in item:
                continue
            fields_present.add(field)
            results[field]["total"] += 1
            if exact_match_hit(item[field], truth):
                results[field]["hit"] += 1
    return results, fields_present

def discover_fields(records):
    discovered = set()
    for item in records:
        for key in item.keys():
            if key.startswith("line_"):
                discovered.add(key)
    return discovered

roots = [
    Path("dataset_split/outputs/qwen/"),
    Path("dataset_split/outputs/qwen/rerank"),
]

rows = []
skipped = []
all_fields = set()

json_files = []
for root in roots:
    if not root.exists():
        print(f"skip missing dir: {root}")
        continue
    json_files.extend(sorted(root.rglob("*.json")))

cached_records = {}
for json_path in json_files:
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        skipped.append((json_path.as_posix(), f"json parse error: {e}"))
        continue

    records = extract_records(data)
    if not records:
        skipped.append((json_path.as_posix(), "non-record-list json shape"))
        continue

    cached_records[json_path] = records
    all_fields.update(discover_fields(records))

fields = sorted(all_fields)
overall = {field: {"hit": 0, "total": 0} for field in fields}
field_coverage_files = {field: 0 for field in fields}
field_coverage_records = {field: 0 for field in fields}

for json_path, records in cached_records.items():
    file_results, fields_present = evaluate_file(records, fields)
    for field in fields:
        overall[field]["hit"] += file_results[field]["hit"]
        overall[field]["total"] += file_results[field]["total"]
        if field in fields_present:
            field_coverage_files[field] += 1
            field_coverage_records[field] += file_results[field]["total"]
    rows.append((json_path, file_results, fields_present))

summary_path = Path("produce/mergebert_20/buchong_exact_match.md")
summary_path.parent.mkdir(parents=True, exist_ok=True)

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("# produce/mergebert_20 精准匹配汇总\n\n")
    f.write("## 扫描目录\n\n")
    for root in roots:
        f.write(f"- {root.as_posix()}\n")

    f.write("\n## 统计字段（自动识别）\n\n")
    if fields:
        for field in fields:
            f.write(f"- {field}\n")
    else:
        f.write("- 未发现 line_* 字段\n")

    f.write("\n## 字段覆盖情况\n\n")
    f.write("| Field | Files Covered | Records Covered |\n")
    f.write("| --- | --- | --- |\n")
    for field in fields:
        f.write(
            f"| {field} | {field_coverage_files[field]} | {field_coverage_records[field]} |\n"
        )

    f.write("\n## 单文件结果（仅展示该文件实际存在字段）\n\n")
    f.write("| File | Field | Hit | Total | Accuracy |\n")
    f.write("| --- | --- | --- | --- | --- |\n")
    for file_path, file_results, fields_present in rows:
        rel_path = file_path.as_posix()
        for field in sorted(fields_present):
            hit = file_results[field]["hit"]
            total = file_results[field]["total"]
            acc = (hit / total) if total else 0
            f.write(f"| {rel_path} | {field} | {hit} | {total} | {acc:.4f} |\n")

    f.write("\n## 总体结果（按字段实际出现记录统计）\n\n")
    f.write("| Field | Hit | Total | Accuracy |\n")
    f.write("| --- | --- | --- | --- |\n")
    for field in fields:
        hit = overall[field]["hit"]
        total = overall[field]["total"]
        acc = (hit / total) if total else 0
        f.write(f"| {field} | {hit} | {total} | {acc:.4f} |\n")

    if skipped:
        f.write("\n## 跳过文件\n\n")
        f.write("| File | Reason |\n")
        f.write("| --- | --- |\n")
        for path, reason in skipped:
            f.write(f"| {path} | {reason} |\n")

print(f"summary saved -> {summary_path}")
print(f"fields discovered -> {len(fields)}")
print(f"json files counted -> {len(rows)}")
print(f"json files skipped -> {len(skipped)}")

In [ ]:
import json
import math
import re
from collections import Counter, defaultdict
from pathlib import Path

def tokenize(text: str):
    if text is None:
        return []
    s = str(text).strip()
    if not s:
        return []
    # 把“单词/数字”和“标点符号”分开，适合代码片段
    return re.findall(r"\w+|[^\w\s]", s, flags=re.UNICODE)

def ngram_counter(tokens, n: int) -> Counter:
    if n <= 0 or len(tokens) < n:
        return Counter()
    return Counter(tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1))

def clipped_counts(candidate_tokens, reference_tokens, n: int):
    cand = ngram_counter(candidate_tokens, n)
    ref = ngram_counter(reference_tokens, n)
    matches = 0
    total = 0
    for ng, c in cand.items():
        total += c
        matches += min(c, ref.get(ng, 0))
    return matches, total

def sentence_bleu4(candidate: str, reference: str, smooth: float = 1.0) -> float:
    cand_tokens = tokenize(candidate)
    ref_tokens = tokenize(reference)
    c = len(cand_tokens)
    r = len(ref_tokens)
    if c == 0:
        return 0.0

    # modified n-gram precisions
    log_p_sum = 0.0
    for n in range(1, 5):
        m, t = clipped_counts(cand_tokens, ref_tokens, n)
        # add-one smoothing，避免 0
        p = (m + smooth) / (t + smooth) if t > 0 else (smooth / (smooth + 1.0))
        log_p_sum += 0.25 * math.log(p)

    geo_mean = math.exp(log_p_sum)
    bp = 1.0 if c > r else math.exp(1.0 - (r / c))
    return bp * geo_mean

def corpus_bleu4(pairs_tokens, smooth: float = 1.0) -> float:
    matches = [0, 0, 0, 0]
    totals = [0, 0, 0, 0]
    c_len = 0
    r_len = 0
    for cand_tokens, ref_tokens in pairs_tokens:
        c_len += len(cand_tokens)
        r_len += len(ref_tokens)
        for n in range(1, 5):
            m, t = clipped_counts(cand_tokens, ref_tokens, n)
            matches[n - 1] += m
            totals[n - 1] += t

    if c_len == 0:
        return 0.0

    log_p_sum = 0.0
    for n in range(1, 5):
        m = matches[n - 1]
        t = totals[n - 1]
        p = (m + smooth) / (t + smooth) if t > 0 else 0.0
        if p <= 0:
            return 0.0
        log_p_sum += 0.25 * math.log(p)

    geo_mean = math.exp(log_p_sum)
    bp = 1.0 if c_len > r_len else math.exp(1.0 - (r_len / c_len))
    return bp * geo_mean

def normalize_candidate_list(v):
    if isinstance(v, list):
        return [str(x).strip() for x in v if str(x).strip()]
    if v is None:
        return []
    s = str(v).strip()
    return [s] if s else []

target_path = Path("dataset_split/outputs/qwen/mergebert_20_qwen.json")
reference_key = "res_region"
summary_path = target_path.with_name(target_path.stem + "_bleu4.md")

with open(target_path, "r", encoding="utf-8") as f:
    data = json.load(f)

if not isinstance(data, list):
    raise ValueError(f"expected list in {target_path}, got {type(data)}")

# 自动发现候选字段：key 以 line 开头，且值为 list（或至少在部分样本中为 list）
field_counts = defaultdict(int)
field_list_counts = defaultdict(int)
for item in data:
    if not isinstance(item, dict):
        continue
    for k, v in item.items():
        if not str(k).startswith("line"):
            continue
        field_counts[k] += 1
        if isinstance(v, list):
            field_list_counts[k] += 1

candidate_fields = sorted([k for k in field_counts.keys() if field_list_counts[k] > 0])

def evaluate_field(field: str):
    top1_pairs = []
    best_pairs = []
    top1_sentence_scores = []
    best_sentence_scores = []

    total_items = 0
    missing_ref = 0
    missing_cand = 0

    for item in data:
        if not isinstance(item, dict):
            continue
        total_items += 1
        ref = str(item.get(reference_key, "") or "").strip()
        if not ref:
            missing_ref += 1
            continue

        cand_list = normalize_candidate_list(item.get(field))
        if not cand_list:
            missing_cand += 1
            continue

        # top1
        cand_top1 = cand_list[0]
        cand_top1_tokens = tokenize(cand_top1)
        ref_tokens = tokenize(ref)
        top1_pairs.append((cand_top1_tokens, ref_tokens))
        top1_sentence_scores.append(sentence_bleu4(cand_top1, ref))

        # best-of-k（oracle）
        best_cand = max(cand_list, key=lambda c: sentence_bleu4(c, ref))
        best_pairs.append((tokenize(best_cand), ref_tokens))
        best_sentence_scores.append(sentence_bleu4(best_cand, ref))

    top1_corpus = corpus_bleu4(top1_pairs)
    best_corpus = corpus_bleu4(best_pairs)
    top1_avg = sum(top1_sentence_scores) / len(top1_sentence_scores) if top1_sentence_scores else 0.0
    best_avg = sum(best_sentence_scores) / len(best_sentence_scores) if best_sentence_scores else 0.0

    return {
        "field": field,
        "items": len(top1_pairs),
        "missing_ref": missing_ref,
        "missing_cand": missing_cand,
        "top1_corpus_bleu4": top1_corpus,
        "top1_avg_sentence_bleu4": top1_avg,
        "best_corpus_bleu4": best_corpus,
        "best_avg_sentence_bleu4": best_avg,
    }

results = [evaluate_field(field) for field in candidate_fields]

print(f"source: {target_path}")
print(f"records: {len(data)}")
print(f"candidate fields: {len(candidate_fields)} -> {candidate_fields}")
print()
for r in results:
    print(
        f"{r['field']}: items={r['items']} | "
        f"BLEU4(top1 corpus)={r['top1_corpus_bleu4']:.4f} "
        f"avg={r['top1_avg_sentence_bleu4']:.4f} | "
        f"BLEU4(best corpus)={r['best_corpus_bleu4']:.4f} "
        f"avg={r['best_avg_sentence_bleu4']:.4f}"
    )

# 写 markdown 汇总（便于留档）
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(f"# BLEU-4 评估汇总\n\n")
    f.write(f"- source: {target_path.as_posix()}\n")
    f.write(f"- records: {len(data)}\n")
    f.write(f"- reference: {reference_key}\n")
    f.write(f"- candidate fields: {', '.join(candidate_fields) if candidate_fields else '(none)'}\n\n")
    f.write("| Field | Items | BLEU4 Top1 (Corpus) | BLEU4 Top1 (Avg) | BLEU4 Best (Corpus) | BLEU4 Best (Avg) |\n")
    f.write("| --- | --- | --- | --- | --- | --- |\n")
    for r in results:
        f.write(
            "| {field} | {items} | {t1c:.4f} | {t1a:.4f} | {bc:.4f} | {ba:.4f} |\n".format(
                field=r["field"],
                items=r["items"],
                t1c=r["top1_corpus_bleu4"],
                t1a=r["top1_avg_sentence_bleu4"],
                bc=r["best_corpus_bleu4"],
                ba=r["best_avg_sentence_bleu4"],
            )
        )

print()
print(f"summary saved -> {summary_path}")

In [ ]:
import json
import re
from pathlib import Path

base = Path('dataset_split/outputs/qwen/rerank')
repo_stats = {}

for p in sorted(base.glob('*.json')):
    if 'summary' in p.name or 'topk' in p.name:
        continue
    try:
        data = json.loads(p.read_text(encoding='utf-8'))
    except Exception:
        continue
    if not isinstance(data, list):
        continue

    m = re.search(r'([A-Za-z0-9_]+)_time20', p.name)
    if not m:
        continue
    repo = m.group(1)
    if repo == 'mergebert':
        continue

    for item in data:
        if not isinstance(item, dict) or 'line_withRefer1' not in item:
            continue
        truth = str(item.get('res_region', '')).strip()
        cand = item.get('line_withRefer1')
        if isinstance(cand, list):
            ok = any(str(x).strip() == truth for x in cand if str(x).strip())
        else:
            ok = str(cand).strip() == truth
        stats = repo_stats.setdefault(repo, {'hit': 0, 'total': 0})
        stats['total'] += 1
        if ok:
            stats['hit'] += 1

for repo in sorted(repo_stats):
    h = repo_stats[repo]['hit']
    t = repo_stats[repo]['total']
    acc = h / t if t else 0
    print(f'{repo}\t{h}\t{t}\t{acc:.4f}')

In [ ]:
import json
from pathlib import Path
from collections import defaultdict


def main():
    file_path = Path("dataset_split/outputs/qwen/rerank/mergebert_20_qwen_buchong_sorted.json")
    if not file_path.exists():
        print(f"文件不存在: {file_path}")
        return

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        print("JSON 根节点不是列表，请检查文件格式。")
        return

    # repo -> {"top1": hit, "top3": hit, "total": n}
    repo_stats = defaultdict(lambda: {"top1": 0, "top3": 0, "total": 0})

    for item in data:
        if not isinstance(item, dict):
            continue

        repo = item.get("repo", "unknown")
        truth = str(item.get("res_region", "")).strip()
        if not truth:
            continue  # 跳过空正确答案

        cand_list = item.get("line_withRefer1")
        if not isinstance(cand_list, list):
            cand_list = [cand_list] if cand_list is not None else []

        # 统一去空白后比较
        norm_cands = [str(c).strip() for c in cand_list]

        top1_ok = len(norm_cands) >= 1 and norm_cands[0] == truth
        top3_ok = any(c == truth for c in norm_cands[:3])

        stats = repo_stats[repo]
        stats["total"] += 1
        if top1_ok:
            stats["top1"] += 1
        if top3_ok:
            stats["top3"] += 1

    # 打印表头
    print(f"{'Repo':<60} {'Top1':>8} {'Top3':>8} {'Total':>8} {'Acc@1':>8} {'Acc@3':>8}")
    print("-" * 110)

    total_all = {"top1": 0, "top3": 0, "total": 0}
    for repo in sorted(repo_stats.keys()):
        s = repo_stats[repo]
        t = s["total"]
        acc1 = s["top1"] / t if t else 0.0
        acc3 = s["top3"] / t if t else 0.0
        print(f"{repo:<60} {s['top1']:>8} {s['top3']:>8} {t:>8} {acc1:>8.4f} {acc3:>8.4f}")

        total_all["top1"] += s["top1"]
        total_all["top3"] += s["top3"]
        total_all["total"] += s["total"]

    # 汇总
    if total_all["total"] > 0:
        print("-" * 110)
        acc1 = total_all["top1"] / total_all["total"]
        acc3 = total_all["top3"] / total_all["total"]
        print(f"{'ALL':<60} {total_all['top1']:>8} {total_all['top3']:>8} "
              f"{total_all['total']:>8} {acc1:>8.4f} {acc3:>8.4f}")


if __name__ == "__main__":
    main()